In [1]:
library(tidyverse)
library(GGally)
library(car)
library(broom)

── Attaching core tidyverse packages ──────────────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: carData


Attaching package: ‘car’


The following object is masked from ‘package:dplyr’:

    recode


The following object is masked from ‘package:purrr’:

    some




In [2]:
players_data <- read_csv("players.csv") %>%
    mutate(pos = factor(sub(",.*", "", position), levels = c("GK", "DF", "MF", "FW"))) %>%
    filter(!is.na(goals)) %>%
    filter(position != "GK") %>%
    mutate(hours = minutes/60) %>%
    select(-position, -team_country, -minutes, -shots_on_target)

head(players_data)

Rows: 1248 Columns: 13
── Column specification ────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr  (2): team_country, position
dbl (11): age, minutes, goals, assists, cards_yellow, shots, shots_on_target...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


age,goals,assists,cards_yellow,shots,fouls,offsides,interceptions,tackles_won,pos,hours
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<dbl>
23,0,0,0,0,0,1,0,0,FW,0.31666667
26,1,0,0,3,1,0,0,1,FW,4.18333333
24,0,0,0,2,0,0,1,0,MF,1.63333333
34,0,0,0,0,0,0,4,3,DF,6.00000000
23,0,0,1,9,3,1,3,5,MF,6.00000000
23,0,0,0,0,0,0,0,0,FW,0.01666667


In [3]:
players_data_lin_reg <- players_data %>%
    filter(goals > 0)

In [8]:
players_data_log_reg <- players_data %>%
    mutate(scored=if_else(goals > 0, 1, 0)) %>%
    select(-goals)

In [15]:
lin_reg_model = step(lm(log(goals) ~ ., data=players_data_lin_reg), direction = "backward")
log_reg_model = step(glm(scored ~ ., data=players_data_log_reg, family=binomial), direction = "backward")

Start:  AIC=-324.51
log(goals) ~ age + assists + cards_yellow + shots + fouls + offsides + 
    interceptions + tackles_won + pos + hours

                Df Sum of Sq    RSS     AIC
- offsides       1    0.0029 25.547 -326.49
- age            1    0.0030 25.547 -326.49
- cards_yellow   1    0.0076 25.552 -326.46
- interceptions  1    0.0267 25.571 -326.32
- assists        1    0.0291 25.573 -326.31
- tackles_won    1    0.1887 25.733 -325.19
- pos            2    0.5362 26.080 -324.79
- fouls          1    0.2668 25.811 -324.65
<none>                       25.544 -324.51
- hours          1    0.2914 25.835 -324.48
- shots          1    5.1480 30.692 -293.65

Step:  AIC=-326.49
log(goals) ~ age + assists + cards_yellow + shots + fouls + interceptions + 
    tackles_won + pos + hours

                Df Sum of Sq    RSS     AIC
- age            1    0.0033 25.550 -328.47
- cards_yellow   1    0.0068 25.554 -328.44
- interceptions  1    0.0290 25.576 -328.29
- assists        1    0.0318 

In [16]:
lin_reg_model


Call:
lm(formula = log(goals) ~ shots + pos, data = players_data_lin_reg)

Coefficients:
(Intercept)        shots        posMF        posFW  
  -0.095447     0.053393    -0.004833     0.162888  


In [17]:
log_reg_model


Call:  glm(formula = scored ~ cards_yellow + shots + tackles_won + pos + 
    hours, family = binomial, data = players_data_log_reg)

Coefficients:
 (Intercept)  cards_yellow         shots   tackles_won         posMF  
     -3.3771        0.2991        0.3431       -0.0894        0.5374  
       posFW         hours  
      0.7325        0.1085  

Degrees of Freedom: 976 Total (i.e. Null);  970 Residual
Null Deviance:	    930.6 
Residual Deviance: 679.9 	AIC: 693.9

In [18]:
lin_reg_results <- 
   tidy(lin_reg_model, conf.int=0.95) %>% 
   mutate_if(is.numeric, round, 2)
lin_reg_results

term,estimate,std.error,statistic,p.value,conf.low,conf.high
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
(Intercept),-0.10,0.08,-1.27,0.21,-0.24,0.05
shots,0.05,0.01,10.63,0.00,0.04,0.06
posMF,0.00,0.09,-0.06,0.95,-0.17,0.16
posFW,0.16,0.09,1.74,0.08,-0.02,0.35


In [19]:
log_reg_results <- 
   tidy(log_reg_model, conf.int=0.95) %>% 
   mutate_if(is.numeric, round, 2)
log_reg_results

term,estimate,std.error,statistic,p.value,conf.low,conf.high
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
(Intercept),-3.38,0.30,-11.38,0.00,-3.99,-2.82
cards_yellow,0.30,0.18,1.64,0.10,-0.06,0.65
shots,0.34,0.04,7.95,0.00,0.26,0.43
tackles_won,-0.09,0.05,-1.76,0.08,-0.19,0.01
posMF,0.54,0.28,1.95,0.05,0.00,1.09
posFW,0.73,0.33,2.25,0.02,0.10,1.38
hours,0.11,0.06,1.74,0.08,-0.01,0.23
